# RAGAS EVALUATION

In [ ]:
import os
import json
import pandas 
import dotenv
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_correctness,
    context_precision,
    context_recall
)

c:\Users\816032048\Documents\GitHub\Enterprise-Knowledge-Mining-System\.venv-ragas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\816032048\AppData\Local\Temp\ipykernel_3100\1286080087.py:16: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\816032048\AppData\Local\Temp\ipykernel_3100\1286080087.py:16: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\816032048\AppData\Local\Temp\ipykernel_31

In [2]:
# nest_asyncio.apply()
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")

print("API key loaded:", os.getenv("OPENAI_API_KEY") is not None)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_KEY")

with open("rag_evaluation_scored_results.json", "r", encoding="utf-8") as f:
    rag_eval_results = json.load(f)

print(f"Loaded {len(rag_eval_results)} evaluation results")
print(rag_eval_results[0].keys())

API key loaded: True
Loaded 75 evaluation results
dict_keys(['question', 'expected_answer', 'generated_answer', 'source_title', 'expected_keywords', 'difficulty', 'retrieved_sources', 'retrieved_contexts', 'source_details', 'source_retrieved_top10', 'keyword_coverage', 'matched_keywords', 'answer_similarity'])


In [3]:
ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for item in rag_eval_results:
    contexts = item.get("retrieved_contexts", [])

    if isinstance(contexts, str):
        contexts = [contexts]

    ragas_data["question"].append(item["question"])
    ragas_data["answer"].append(item["generated_answer"])
    ragas_data["contexts"].append(contexts)
    ragas_data["ground_truth"].append(item["expected_answer"])

ragas_dataset = Dataset.from_dict(ragas_data)

print(ragas_dataset)
print(ragas_dataset[0])

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 75
})
{'question': 'What problem does the paper address in creating semantic segmentation datasets?', 'answer': 'The paper addresses the problem that creating semantic segmentation datasets is highly laborious and time-consuming because every pixel in an image must be accurately annotated. This annotation process is a major bottleneck, requiring many hours of manual labeling, often by specialized experts, especially in medical imaging contexts.', 'contexts': ['The fundamental problem of current research is that there is no longer enough data for advanced algorithms or models for special applications. Coupling custom datasets and DL models will be the future theme to many research papers. So many researchers’ outputs consist of not only algorithms or architectures, but also datasets or methods to amass data. Dataset annotation is a major bottleneck in the DL workflow which requires many hours of ma

In [4]:
ragas_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    timeout=60,
    max_retries=2
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [5]:
small_dataset = ragas_dataset.select(range(3))

test_scores = evaluate(
    small_dataset,
    metrics=[
        faithfulness,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)

test_df = test_scores.to_pandas()
display(test_df)

Evaluating: 100%|██████████| 12/12 [00:29<00:00,  2.47s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_precision,context_recall
0,What problem does the paper address in creatin...,[The fundamental problem of current research i...,The paper addresses the problem that creating ...,The paper addresses the high cost and human ef...,1.000000,0.328666,1.0,1.0
1,What approach is proposed for generating groun...,[Large amounts of high-quality annotated sampl...,An approach proposed for generating ground tru...,The paper proposes creating pixel-accurate sem...,0.777778,0.106930,0.0,0.0
2,What result did the authors report using game ...,[We begin with experiments on the CamVid datas...,The authors reported that using synthetic game...,Models trained with game data and only one-thi...,1.000000,0.459618,1.0,1.0


In [6]:
ragas_scores = evaluate(
    ragas_dataset,
    metrics=[
        faithfulness,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    raise_exceptions=False
)

ragas_df = ragas_scores.to_pandas()
display(ragas_df)

Evaluating: 100%|██████████| 300/300 [12:59<00:00,  2.60s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness,context_precision,context_recall
0,What problem does the paper address in creatin...,[The fundamental problem of current research i...,The paper addresses the problem that creating ...,The paper addresses the high cost and human ef...,1.000000,0.328666,1.0,1.0
1,What approach is proposed for generating groun...,[Large amounts of high-quality annotated sampl...,An approach proposed for generating ground tru...,The paper proposes creating pixel-accurate sem...,0.777778,0.106930,0.0,0.0
2,What result did the authors report using game ...,[We begin with experiments on the CamVid datas...,The authors reported that using synthetic game...,Models trained with game data and only one-thi...,1.000000,0.459618,1.0,1.0
3,What is the main purpose of the proposed HCI s...,[One of the promising fields in artificial int...,The main purpose of the proposed HCI system is...,The system uses face and hand gesture recognit...,1.000000,0.135797,0.0,0.0
4,What techniques are used to extract the face a...,"[In the proposed technique, first, the hand ge...",The techniques used to extract the face and ha...,The method combines skin detection and cascade...,1.000000,0.185618,1.0,1.0
...,...,...,...,...,...,...,...,...
70,What folktale was selected for the animation a...,[People tend to better understand any message ...,The selected folktale for the animation was a ...,The story Ijapa ati Atioro was selected becaus...,1.000000,0.456517,0.0,0.0
71,What software tools were used to develop the a...,[People tend to better understand any message ...,The software tools used to develop the animati...,The animation was developed using Maya Autodes...,1.000000,0.721918,1.0,1.0
72,What is the main problem studied in this thesis?,[Chapter 3 reviews literature that is related ...,The main problem studied in this thesis is add...,The thesis studies methods for computing and o...,1.000000,0.101431,0.0,0.0
73,What two wireless models are used to study cov...,[In this thesis we present new methods for the...,The thesis studies coverage computation using ...,The thesis studies coverage computation using ...,1.000000,0.983064,1.0,1.0


In [7]:
print("RAGAS Evaluation Summary")
print("=" * 50)

ragas_metric_cols = [
    "faithfulness",
    "answer_correctness",
    "context_precision",
    "context_recall"
]

for col in ragas_metric_cols:
    print(f"{col}: {ragas_df[col].mean():.2%}")

RAGAS Evaluation Summary
faithfulness: 89.16%
answer_correctness: 50.79%
context_precision: 64.33%
context_recall: 63.33%


In [ ]:
ragas_df.to_csv("ragas_evaluation_results.csv", index=False)
print("Saved ragas_evaluation_results.csv")

Saved ragas_evaluation_results.csv
